# WHOOP Data Cleaning Pipeline


This notebook turns raw exports produced by `fetch_whoop_data.py` into two tidy files:

1. **`clean_daily.csv`** – day‑level aggregate (Cycles + Sleep + Recovery)
2. **`clean_workout.csv`** – trimmed workout log with intuitive column names and sport labels

Feel free to adapt paths or add extra features.

## 1 · Setup – imports & paths

In [9]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('user_1_raw_exports')  # folder with raw CSVs
OUT_DIR = Path('user_1_clean')               


## 2 · Load raw exports

In [10]:
cycles   = pd.read_csv(next(RAW_DIR.glob('cycles_*.csv')))
sleep    = pd.read_csv(next(RAW_DIR.glob('sleep_*.csv')))
recovery = pd.read_csv(next(RAW_DIR.glob('recovery_*.csv')))
workout  = pd.read_csv(next(RAW_DIR.glob('workout_*.csv')))

print('Loaded:', len(cycles), 'cycles', '/', len(workout), 'workouts')


Loaded: 23 cycles / 5 workouts


## 3 · Drop empty rows / columns

In [11]:
def drop_empty(df):
    df = df.dropna(how='all')
    return df.loc[:, ~df.isna().all()]

cycles   = drop_empty(cycles)
sleep    = drop_empty(sleep)
recovery = drop_empty(recovery)
workout  = drop_empty(workout)


## 4 · Clean workout table

In [12]:
# Columns to keep
keep_cols = [
    'user_id','id','start','end','sport_id',
    'score.strain','score.average_heart_rate','score.max_heart_rate','score.kilojoule',
    'score.zone_duration.zone_zero_milli','score.zone_duration.zone_one_milli',
    'score.zone_duration.zone_two_milli','score.zone_duration.zone_three_milli',
    'score.zone_duration.zone_four_milli','score.zone_duration.zone_five_milli'
]
workout = workout[keep_cols]

# Rename columns
workout = workout.rename(columns={
    'id':'workout_id',
    'sport_id':'sport',
    'score.strain':'strain',
    'score.average_heart_rate':'avg_hr',
    'score.max_heart_rate':'max_hr',
    'score.kilojoule':'kilojoule',
    'score.zone_duration.zone_zero_milli':'zone0_ms',
    'score.zone_duration.zone_one_milli':'zone1_ms',
    'score.zone_duration.zone_two_milli':'zone2_ms',
    'score.zone_duration.zone_three_milli':'zone3_ms',
    'score.zone_duration.zone_four_milli':'zone4_ms',
    'score.zone_duration.zone_five_milli':'zone5_ms'
})


### Map sport IDs → names

In [13]:
sport_map = {
 -1:'Activity',0:'Running',1:'Cycling',16:'Baseball',17:'Basketball',18:'Rowing',
 19:'Fencing',20:'Field Hockey',21:'Football',22:'Golf',24:'Ice Hockey',
 25:'Lacrosse',27:'Rugby',28:'Sailing',29:'Skiing',30:'Soccer',31:'Softball',
 32:'Squash',33:'Swimming',34:'Tennis',35:'Track & Field',36:'Volleyball',
 37:'Water Polo',38:'Wrestling',39:'Boxing',42:'Dance',43:'Pilates',44:'Yoga',
 45:'Weightlifting',47:'Cross Country Skiing',48:'Functional Fitness',
 49:'Duathlon',51:'Gymnastics',52:'Hiking/Rucking',53:'Horseback Riding',
 55:'Kayaking',56:'Martial Arts',57:'Mountain Biking',59:'Powerlifting',
 60:'Rock Climbing',61:'Paddleboarding',62:'Triathlon',63:'Walking',
 64:'Surfing',65:'Elliptical',66:'Stairmaster',70:'Meditation',71:'Other',
 73:'Diving',74:'Operations - Tactical',75:'Operations - Medical',
 76:'Operations - Flying',77:'Operations - Water',82:'Ultimate',83:'Climber',
 84:'Jumping Rope',85:'Australian Football',86:'Skateboarding',87:'Coaching',
 88:'Ice Bath',89:'Commuting',90:'Gaming',91:'Snowboarding',92:'Motocross',
 93:'Caddying',94:'Obstacle Course Racing',95:'Motor Racing',96:'HIIT',
 97:'Spin',98:'Jiu Jitsu',99:'Manual Labor',100:'Cricket',101:'Pickleball',
 102:'Inline Skating',103:'Box Fitness',104:'Spikeball',105:'Wheelchair Pushing',
 106:'Paddle Tennis',107:'Barre',108:'Stage Performance',109:'High Stress Work',
 110:'Parkour',111:'Gaelic Football',112:'Hurling/Camogie',113:'Circus Arts',
 121:'Massage Therapy',123:'Strength Trainer',125:'Watching Sports',
 126:'Assault Bike',127:'Kickboxing',128:'Stretching',230:'Table Tennis',
 231:'Badminton',232:'Netball',233:'Sauna',234:'Disc Golf',235:'Yard Work',
 236:'Air Compression',237:'Percussive Massage',238:'Paintball',239:'Ice Skating',
 240:'Handball',248:'F45 Training',249:'Padel',250:"Barry's",251:'Dedicated Parenting',
 252:'Stroller Walking',253:'Stroller Jogging',254:'Toddlerwearing',255:'Babywearing',
 258:'Barre3',259:'Hot Yoga',261:'Stadium Steps',262:'Polo',263:'Musical Performance',
 264:'Kite Boarding',266:'Dog Walking',267:'Water Skiing',268:'Wakeboarding',
 269:'Cooking',270:'Cleaning',272:'Public Speaking'
}


workout['sport'] = workout['sport'].map(sport_map).fillna('Unknown')
workout.head()

,user_id,workout_id,start,end,sport,strain,avg_hr,max_hr,kilojoule,zone0_ms,zone1_ms,zone2_ms,zone3_ms,zone4_ms,zone5_ms
0,14645583,1650746156,2025-04-27 13:55:30.096,2025-04-27 14:23:45.801,Soccer,10.6184,140,182,1271.46400,58678,277822,530607,422980,363361,42296
1,14645583,1642617980,2025-04-23 20:37:30.612,2025-04-23 22:07:29.508,Soccer,12.1838,118,169,2507.69560,1030564,2506151,1017070,590247,183608,0
2,14645583,1640341665,2025-04-22 20:28:30.497,2025-04-22 22:04:29.656,Soccer,12.1825,119,172,2679.97270,790231,3126165,988212,690205,164385,0
3,14645583,1638057882,2025-04-21 21:45:00.747,2025-04-21 22:02:59.335,Soccer,4.9929,109,144,354.61084,243209,820037,15381,0,0,0
4,14645583,1627408707,2025-04-16 20:25:30.060,2025-04-16 20:50:59.503,Soccer,5.2941,107,139,463.69290,377795,1121885,29801,0,0,0


## 5 · Build clean daily table

In [14]:
# Align key names
sleep = sleep.rename(columns={'cycle_id':'id'})
recovery = recovery.rename(columns={'cycle_id':'id'})

# Merge
daily = cycles.merge(sleep, on='id', how='left', suffixes=('','_sleep'))
daily = daily.merge(recovery, on='id', how='left', suffixes=('','_rec'))

# Drop all-null cols
daily = daily.loc[:, ~daily.isna().all()]

# Rename some common metrics for readability (optional, extend to taste)
rename_map = {
    'score.strain':'strain',
    'score.average_heart_rate':'avg_hr',
    'score.kilojoule':'kilojoule',
    'score.recovery_score':'recovery_score',
    'score.resting_heart_rate':'rhr',
    'score.hrv_rmssd_milli':'hrv_ms',
    'score.performance_percentage':'sleep_perf_pct',
    'score.stage_summary.total_in_bed_time_milli':'time_in_bed_ms',
    'score.stage_summary.total_sleep_time_milli':'sleep_ms'
}
daily = daily.rename(columns={k:v for k,v in rename_map.items() if k in daily.columns})

daily.head()


,user_id,created_at,updated_at,timezone_offset,score_state,id,start,end,strain,kilojoule,...,created_at_rec,updated_at_rec,score_state_rec,sleep_id,score.user_calibrating,recovery_score,rhr,hrv_ms,score.spo2_percentage,score.skin_temp_celsius
0,14645583,2025-04-28 06:33:57.081,2025-04-28 09:54:27.827,+02:00,SCORED,880908379,2025-04-28 01:21:19.795,NaN,4.427445,3464.3535,...,2025-04-28 04:33:57.081,2025-04-28 07:54:32.488,SCORED,1.651849e+09,False,50.0,56.0,55.432358,94.975000,34.600000
1,14645583,2025-04-27 11:29:36.854,2025-04-28 06:34:02.199,+02:00,SCORED,880107425,2025-04-27 00:00:00.000,2025-04-28 01:21:19.795,13.855994,7131.6562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,14645583,2025-04-27 11:29:36.604,2025-04-27 11:29:41.972,+02:00,SCORED,880107422,2025-04-25 01:26:27.013,2025-04-27 00:00:00.000,4.074928,2074.7407,...,2025-04-27 09:29:36.604,2025-04-27 09:29:41.025,SCORED,1.650133e+09,False,38.0,52.0,58.028440,94.633330,33.768333
3,14645583,2025-04-27 11:13:49.151,2025-04-27 11:29:41.708,+02:00,SCORED,880094111,2025-04-24 02:38:58.846,2025-04-25 01:26:27.013,10.526950,8844.5910,...,2025-04-27 09:13:49.151,2025-04-27 09:13:53.819,SCORED,1.650110e+09,False,56.0,59.0,67.561000,97.210526,34.233334
4,14645583,2025-04-23 09:26:34.421,2025-04-27 11:13:54.275,+02:00,SCORED,875547576,2025-04-23 02:51:30.471,2025-04-24 02:38:58.846,14.866322,11360.8930,...,2025-04-23 07:26:34.421,2025-04-23 07:26:39.371,SCORED,1.641267e+09,False,34.0,49.0,51.024994,95.038460,34.406666


## 6 · Save cleaned CSVs

In [15]:
OUT_DIR.mkdir(exist_ok=True)
daily_path   = OUT_DIR / 'clean_daily.csv'
workout_path = OUT_DIR / 'clean_workout.csv'

daily.to_csv(daily_path, index=False)
workout.to_csv(workout_path, index=False)

print('Wrote', daily_path, 'and', workout_path)


Wrote user_1_clean/clean_daily.csv and user_1_clean/clean_workout.csv
